In [ ]:
import sys
from pathlib import Path

project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))
import os
os.chdir(project_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

In [ ]:
# Load the main 2015-2020 prices and the FinBERT daily sentiment, then merge.
from src.data.loader import load_and_filter_dataset
from src.data.news_sentiment import load_daily_sentiment
from src.utils.config import load_config

config = load_config()
prices = load_and_filter_dataset(config=config)
prices['date'] = pd.to_datetime(prices['date'])
sent = load_daily_sentiment(config=config)

prices = prices.sort_values(['symbol', 'date'])
# same-day return (close-to-close reaction) and next-day forward return
prices['ret_same'] = np.log(prices['close'] / prices.groupby('symbol')['close'].shift(1))
prices['ret_fwd'] = np.log(prices.groupby('symbol')['close'].shift(-1) / prices['close'])

df = prices.merge(sent, on=['symbol', 'date'], how='inner')
df = df.dropna(subset=['ret_same', 'ret_fwd', 'news_compound']).reset_index(drop=True)

print('rows (price + news days):', len(df))
print('symbols:', df['symbol'].nunique())
print('date range:', df['date'].min().date(), '->', df['date'].max().date())
print('total articles:', int(df['news_count'].sum()))
df.head()

## 1. News coverage over time

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

monthly = df.groupby(df['date'].dt.to_period('M'))['news_count'].sum()
monthly.index = monthly.index.to_timestamp()
axes[0].bar(monthly.index, monthly.values, width=20, color='#3b82f6')
axes[0].set_title('Articles per month')
axes[0].set_ylabel('articles')

per_ticker = df.groupby('symbol')['news_count'].sum().sort_values(ascending=False)
axes[1].bar(per_ticker.index, per_ticker.values, color='#3b82f6')
axes[1].set_title('Articles per ticker')
axes[1].set_ylabel('articles')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 2. Sentiment distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].hist(df['news_compound'], bins=50, color='#3b82f6', edgecolor='black', alpha=0.8)
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_title('Daily compound sentiment distribution')
axes[0].set_xlabel('compound')

axes[1].hist(df['news_pos'], bins=40, alpha=0.6, label='pos', color='#22c55e')
axes[1].hist(df['news_neg'], bins=40, alpha=0.6, label='neg', color='#ef4444')
axes[1].set_title('Positive vs negative scores')
axes[1].legend()

plt.tight_layout()
plt.show()

print(df[['news_compound', 'news_pos', 'news_neg', 'news_neu', 'news_count']].describe().round(4).to_string())

## 3. Same-day impact (the strong signal)

Do days with more positive/negative news coincide with positive/negative market moves *on the same day*?

In [ ]:
# Bin the compound score and look at the mean same-day return per bin.
bins = [-1.01, -0.5, -0.2, -0.05, 0.05, 0.2, 0.5, 1.01]
labels = ['<-0.5', '-0.5..-0.2', '-0.2..-0.05', '-0.05..0.05', '0.05..0.2', '0.2..0.5', '>0.5']
df['sent_bin'] = pd.cut(df['news_compound'], bins=bins, labels=labels)

grp = df.groupby('sent_bin', observed=True)['ret_same'].agg(['mean', 'count'])

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
colors = ['#ef4444' if x < 0 else '#22c55e' for x in grp['mean']]
axes[0].bar(range(len(grp)), grp['mean'].values * 100, color=colors)
axes[0].set_xticks(range(len(grp)))
axes[0].set_xticklabels(grp.index, rotation=45)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title('Mean SAME-DAY return by sentiment bin')
axes[0].set_ylabel('mean return %')

axes[1].scatter(df['news_compound'], df['ret_same'] * 100, alpha=0.15, s=12, color='#3b82f6')
z = np.polyfit(df['news_compound'], df['ret_same'] * 100, 1)
xs = np.linspace(-1, 1, 50)
axes[1].plot(xs, np.polyval(z, xs), 'r-', linewidth=2)
axes[1].set_title('Same-day return vs compound sentiment')
axes[1].set_xlabel('compound')
axes[1].set_ylabel('same-day return %')

plt.tight_layout()
plt.show()

pr = pearsonr(df['news_compound'], df['ret_same'])
sr = spearmanr(df['news_compound'], df['ret_same'])
print(f'Pearson  r = {pr[0]:+.4f}  (p = {pr[1]:.2e})')
print(f'Spearman r = {sr.correlation:+.4f}  (p = {sr.pvalue:.2e})')
print()
print(grp.assign(mean_pct=(grp['mean']*100).round(3)).to_string())

## 4. Next-day predictability (honest result)

Same analysis but for the *next* day. If markets are efficient, news is largely priced in same-day and next-day predictability is weak.

In [ ]:
grp_fwd = df.groupby('sent_bin', observed=True)['ret_fwd'].agg(['mean', 'count'])

colors = ['#ef4444' if x < 0 else '#22c55e' for x in grp_fwd['mean']]
plt.figure(figsize=(8, 5))
plt.bar(range(len(grp_fwd)), grp_fwd['mean'].values * 100, color=colors)
plt.xticks(range(len(grp_fwd)), grp_fwd.index, rotation=45)
plt.axhline(0, color='black', linewidth=0.8)
plt.title('Mean NEXT-DAY return by sentiment bin')
plt.ylabel('mean return %')
plt.tight_layout()
plt.show()

pr = pearsonr(df['news_compound'], df['ret_fwd'])
print(f'compound -> next-day return:  Pearson r = {pr[0]:+.4f}  (p = {pr[1]:.2e})')
print('\nInterpretation: a much weaker (and barely significant) relationship than same-day,')
print('consistent with the efficient-market view that news is absorbed quickly.')

## 5. Event studies: the most extreme news days

Pick the days with the strongest negative and positive aggregated sentiment (with enough articles) and look at the price reaction around them.

In [ ]:
# Require at least a few articles so the day is meaningful.
strong = df[df['news_count'] >= 5].copy()
worst = strong.nsmallest(5, 'news_compound')[['date', 'symbol', 'news_compound', 'news_count', 'ret_same', 'ret_fwd']]
best = strong.nlargest(5, 'news_compound')[['date', 'symbol', 'news_compound', 'news_count', 'ret_same', 'ret_fwd']]

print('=== Most NEGATIVE sentiment days ===')
print(worst.assign(ret_same_pct=(worst['ret_same']*100).round(2), ret_fwd_pct=(worst['ret_fwd']*100).round(2)).drop(columns=['ret_same','ret_fwd']).to_string(index=False))
print('\n=== Most POSITIVE sentiment days ===')
print(best.assign(ret_same_pct=(best['ret_same']*100).round(2), ret_fwd_pct=(best['ret_fwd']*100).round(2)).drop(columns=['ret_same','ret_fwd']).to_string(index=False))

In [ ]:
# Visualize price + sentiment for one ticker over the full window.
plot_ticker = 'NFLX'  # best news coverage in 2015-2020
sub = df[df['symbol'] == plot_ticker].sort_values('date')

fig, ax1 = plt.subplots(figsize=(15, 6))
ax1.plot(sub['date'], sub['close'], color='#111827', linewidth=1.5, label='close')
ax1.set_ylabel('price $')
ax1.set_title(f'{plot_ticker}: price vs daily news sentiment')

ax2 = ax1.twinx()
colors = ['#22c55e' if c > 0 else '#ef4444' for c in sub['news_compound']]
ax2.bar(sub['date'], sub['news_compound'], color=colors, alpha=0.35, width=2)
ax2.set_ylabel('compound sentiment')
ax2.axhline(0, color='gray', linewidth=0.6)

ax1.legend(loc='upper left')
plt.tight_layout()
plt.show()